# 03 Report Tables and Visualizations

This notebook does **not** run model experiments. It reads existing CSV files from `outputs/experiments` and `outputs/predictions`, then creates compact report tables and figures for the cleaned experiment design.

## Setup

In [ ]:
from pathlib import Path
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_EXPERIMENTS = PROJECT_ROOT / "outputs" / "experiments"
OUTPUT_PREDICTIONS = PROJECT_ROOT / "outputs" / "predictions"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "report_figures" / "final_report"
VIDEO_DIR = PROJECT_ROOT / "Charades" / "Charades_v1_480" / "Charades_v1_480"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLIP_MODEL = "openai/clip-vit-base-patch32"
SIGLIP2_MODEL = "google/siglip2-base-patch16-224"

MAIN_EXPERIMENT = "main_ablation_clip"
SIGLIP2_EXPERIMENT = "backbone_siglip2"
OOD_CLIP_EXPERIMENT = "ood1_clip"
OOD_SIGLIP2_EXPERIMENT = "ood1_siglip2"

EXPERIMENTS = {
    MAIN_EXPERIMENT: {"group": "main_ablation", "backbone": "CLIP", "setting": "Normal"},
    SIGLIP2_EXPERIMENT: {"group": "backbone_replacement", "backbone": "SigLIP2", "setting": "Normal"},
    OOD_CLIP_EXPERIMENT: {"group": "novel_location_ood1", "backbone": "CLIP", "setting": "OOD-1"},
    OOD_SIGLIP2_EXPERIMENT: {"group": "novel_location_ood1", "backbone": "SigLIP2", "setting": "OOD-1"},
}

METHOD_ORDER = ["baseline", "query_decomp", "bu_pg", "full"]
METHOD_LABELS = {
    "baseline": "Baseline",
    "query_decomp": "Query Decomp.",
    "bu_pg": "w/o QC-FR",
    "full": "Full",
}
METHOD_COLORS = {
    "baseline": "#3b6ea8",
    "query_decomp": "#4fa99f",
    "bu_pg": "#d95f5f",
    "full": "#6c5ce7",
}
REPORT_METRICS = ["R@1_IoU_0.3", "R@1_IoU_0.5", "R@1_IoU_0.7", "mIoU"]
PAPER_METRICS = ["R@1_IoU_0.5", "R@1_IoU_0.7", "mIoU"]
METRIC_LABELS = {
    "R@1_IoU_0.3": "R@1 IoU=0.3",
    "R@1_IoU_0.5": "R@1 IoU=0.5",
    "R@1_IoU_0.7": "R@1 IoU=0.7",
    "mIoU": "mIoU",
}
SAMPLE_KEYS = ["video_id", "query", "gt_start", "gt_end"]
FRAME_METHODS = ["baseline", "full"]

PROJECT_ROOT, OUTPUT_DIR

## Load Saved Results

In [ ]:
def load_metrics(experiment_names):
    frames = []
    missing = []
    for name in experiment_names:
        path = OUTPUT_EXPERIMENTS / f"{name}_metrics.csv"
        if not path.exists():
            missing.append(path)
            continue
        df = pd.read_csv(path)
        df.insert(0, "experiment", name)
        df.insert(1, "group", EXPERIMENTS[name]["group"])
        df.insert(2, "backbone", EXPERIMENTS[name]["backbone"])
        df.insert(3, "setting", EXPERIMENTS[name]["setting"])
        frames.append(df)
    if missing:
        print("Missing metrics files:")
        for path in missing:
            print(" -", path)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_predictions(experiment_name):
    prediction_dir = OUTPUT_PREDICTIONS / experiment_name
    if not prediction_dir.exists():
        print("Missing prediction directory:", prediction_dir)
        return pd.DataFrame()
    frames = []
    for method in METHOD_ORDER:
        path = prediction_dir / f"{method}.csv"
        if not path.exists():
            continue
        df = pd.read_csv(path)
        if "method" not in df or df["method"].isna().all():
            df["method"] = method
        frames.append(df)
    if not frames:
        print("No prediction CSV files found in", prediction_dir)
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df["method_label"] = df["method"].map(METHOD_LABELS).fillna(df["method"])
    df["gt_length"] = df["gt_end"] - df["gt_start"]
    df["pred_length"] = df["pred_end"] - df["pred_start"]
    return df


def compact_metrics(df):
    if df.empty:
        return df
    out = df[["group", "setting", "backbone", "method"] + REPORT_METRICS].copy()
    out["method"] = out["method"].map(METHOD_LABELS).fillna(out["method"])
    out[REPORT_METRICS] = out[REPORT_METRICS].round(4)
    return out

metrics = load_metrics(EXPERIMENTS.keys())
main_predictions = load_predictions(MAIN_EXPERIMENT)
compact_metrics(metrics)

## Compact Report Tables

In [ ]:
if metrics.empty:
    print("No metrics loaded yet. Run the experiments in notebook 02 first.")
else:
    main_table = compact_metrics(metrics[metrics["experiment"].eq(MAIN_EXPERIMENT)])
    backbone_table = compact_metrics(
        pd.concat([
            metrics[(metrics["experiment"].eq(MAIN_EXPERIMENT)) & (metrics["method"].isin(["baseline", "full"]))],
            metrics[metrics["experiment"].eq(SIGLIP2_EXPERIMENT)],
        ], ignore_index=True)
    )
    ood_table = compact_metrics(metrics[metrics["group"].eq("novel_location_ood1")])

    print("Main ablation")
    display(main_table)
    print("Backbone replacement")
    display(backbone_table)
    print("Novel-location OOD-1")
    display(ood_table)

## Plot Helpers

In [ ]:
def method_label(method):
    return METHOD_LABELS.get(method, method)


def ordered_methods(df):
    available = set(df["method"])
    return [method for method in METHOD_ORDER if method in available]


def apply_axis_style(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=grid_axis, color="#d9d9d9", linewidth=0.8, alpha=0.75)
    ax.set_axisbelow(True)


def save_figure(fig, name):
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=260, bbox_inches="tight")
    print(path)
    return path


def prediction_wide(predictions_df):
    df = predictions_df.drop_duplicates(SAMPLE_KEYS + ["method"])
    wide = df.pivot_table(
        index=SAMPLE_KEYS,
        columns="method",
        values=["iou", "pred_start", "pred_end", "pred_length"],
        aggfunc="first",
    )
    wide.columns = [f"{value}_{method}" for value, method in wide.columns]
    wide = wide.reset_index()
    wide["gt_length"] = wide["gt_end"] - wide["gt_start"]
    return wide


def shorten(text, max_len=82):
    text = str(text)
    return text if len(text) <= max_len else text[: max_len - 3] + "..."

## Figure 1: Main Ablation Metrics

In [ ]:
def plot_main_ablation(metrics_df):
    df = metrics_df[metrics_df["experiment"].eq(MAIN_EXPERIMENT)].copy()
    df = df[df["method"].isin(METHOD_ORDER)]
    df["method"] = pd.Categorical(df["method"], categories=METHOD_ORDER, ordered=True)
    df = df.sort_values("method")
    if df.empty:
        print("Main ablation metrics are missing.")
        return None

    x = np.arange(len(df))
    width = 0.24
    metric_colors = ["#3b6ea8", "#d95f5f", "#4fa99f"]
    fig, ax = plt.subplots(figsize=(9.2, 4.8))
    apply_axis_style(ax)
    for index, metric in enumerate(PAPER_METRICS):
        values = df[metric].to_numpy(dtype=float)
        positions = x + (index - 1) * width
        bars = ax.bar(positions, values, width=width, label=METRIC_LABELS[metric], color=metric_colors[index], alpha=0.84)
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012, f"{value:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([method_label(method) for method in df["method"]], rotation=15, ha="right")
    ax.set_ylim(0, max(0.6, float(df[PAPER_METRICS].max().max()) + 0.12))
    ax.set_ylabel("Score")
    ax.set_title("Main Ablation with CLIP Backbone", weight="bold")
    ax.legend(ncol=3, frameon=False, loc="upper right")
    return fig

fig = plot_main_ablation(metrics)
main_ablation_path = save_figure(fig, "main_ablation_metrics.png") if fig else None
if fig:
    display(fig)
    plt.close(fig)

## Figure 2: CLIP vs SigLIP2 Backbone Replacement

In [ ]:
def plot_backbone_comparison(metrics_df):
    clip_rows = metrics_df[(metrics_df["experiment"].eq(MAIN_EXPERIMENT)) & (metrics_df["method"].isin(["baseline", "full"]))]
    siglip_rows = metrics_df[metrics_df["experiment"].eq(SIGLIP2_EXPERIMENT)]
    df = pd.concat([clip_rows, siglip_rows], ignore_index=True)
    if df.empty:
        print("Backbone comparison metrics are missing.")
        return None
    df["label"] = df["backbone"] + " " + df["method"].map(METHOD_LABELS)
    order = ["CLIP Baseline", "CLIP Full", "SigLIP2 Baseline", "SigLIP2 Full"]
    df["label"] = pd.Categorical(df["label"], categories=order, ordered=True)
    df = df.sort_values("label")

    x = np.arange(len(df))
    width = 0.24
    fig, ax = plt.subplots(figsize=(9.4, 4.8))
    apply_axis_style(ax)
    for index, metric in enumerate(PAPER_METRICS):
        positions = x + (index - 1) * width
        values = df[metric].to_numpy(dtype=float)
        ax.bar(positions, values, width=width, label=METRIC_LABELS[metric], alpha=0.84)
    ax.set_xticks(x)
    ax.set_xticklabels(df["label"], rotation=18, ha="right")
    ax.set_ylim(0, max(0.6, float(df[PAPER_METRICS].max().max()) + 0.12))
    ax.set_ylabel("Score")
    ax.set_title("Backbone Replacement: CLIP vs SigLIP2", weight="bold")
    ax.legend(ncol=3, frameon=False, loc="upper right")
    return fig

fig = plot_backbone_comparison(metrics)
backbone_path = save_figure(fig, "backbone_replacement.png") if fig else None
if fig:
    display(fig)
    plt.close(fig)

## Figure 3: Novel-Location OOD-1

In [ ]:
def plot_ood_comparison(metrics_df):
    df = metrics_df[metrics_df["group"].eq("novel_location_ood1")].copy()
    if df.empty:
        print("OOD-1 metrics are missing.")
        return None
    df = df[df["method"].isin(["baseline", "full"])]
    df["label"] = df["backbone"] + " " + df["method"].map(METHOD_LABELS)
    order = ["CLIP Baseline", "CLIP Full", "SigLIP2 Baseline", "SigLIP2 Full"]
    df["label"] = pd.Categorical(df["label"], categories=order, ordered=True)
    df = df.sort_values("label")

    x = np.arange(len(df))
    width = 0.24
    fig, ax = plt.subplots(figsize=(9.4, 4.8))
    apply_axis_style(ax)
    for index, metric in enumerate(PAPER_METRICS):
        values = df[metric].to_numpy(dtype=float)
        ax.bar(x + (index - 1) * width, values, width=width, label=METRIC_LABELS[metric], alpha=0.84)
    ax.set_xticks(x)
    ax.set_xticklabels(df["label"], rotation=18, ha="right")
    ax.set_ylim(0, max(0.6, float(df[PAPER_METRICS].max().max()) + 0.12))
    ax.set_ylabel("Score")
    ax.set_title("Novel-Location OOD-1, 10s Prefix", weight="bold")
    ax.legend(ncol=3, frameon=False, loc="upper right")
    return fig

fig = plot_ood_comparison(metrics)
ood_path = save_figure(fig, "ood1_backbone_comparison.png") if fig else None
if fig:
    display(fig)
    plt.close(fig)

## Figure 4: Main Ablation IoU Delta

In [ ]:
def plot_iou_delta_vs_baseline(predictions_df):
    if predictions_df.empty:
        print("Main predictions are missing.")
        return None
    wide = prediction_wide(predictions_df)
    if "iou_baseline" not in wide.columns:
        print("Baseline predictions are required for IoU delta.")
        return None
    methods = [method for method in ["query_decomp", "bu_pg", "full"] if f"iou_{method}" in wide.columns]
    if not methods:
        print("No module predictions found for IoU delta.")
        return None
    deltas = [wide[f"iou_{method}"] - wide["iou_baseline"] for method in methods]
    labels = [method_label(method) for method in methods]

    fig, ax = plt.subplots(figsize=(8.8, 4.8))
    apply_axis_style(ax)
    box = ax.boxplot(deltas, labels=labels, patch_artist=True, showfliers=False, widths=0.55)
    for patch, method in zip(box["boxes"], methods):
        patch.set_facecolor(METHOD_COLORS.get(method, "#999999"))
        patch.set_alpha(0.68)
    for median in box["medians"]:
        median.set_color("#111111")
        median.set_linewidth(1.5)
    for index, delta in enumerate(deltas, start=1):
        ax.scatter(index, float(delta.mean()), marker="D", s=42, color="#111111", zorder=4)
        ax.text(index, float(delta.mean()) + 0.035, f"mean {delta.mean():+.2f}", ha="center", fontsize=8)
    ax.axhline(0, color="#111111", linewidth=1.1, linestyle="--")
    ax.set_ylabel("IoU change vs. baseline")
    ax.set_title("Per-Query IoU Change in Main Ablation", weight="bold")
    ax.tick_params(axis="x", rotation=15)
    return fig

fig = plot_iou_delta_vs_baseline(main_predictions)
iou_delta_path = save_figure(fig, "main_iou_delta_vs_baseline.png") if fig else None
if fig:
    display(fig)
    plt.close(fig)

## Figure 5: Single-Sample Walkthrough

In [ ]:
def sample_rows(predictions_df, sample):
    mask = np.ones(len(predictions_df), dtype=bool)
    for column in SAMPLE_KEYS:
        mask &= predictions_df[column].eq(sample[column])
    rows = predictions_df[mask].copy()
    rows = rows[rows["method"].isin(METHOD_ORDER)]
    rows["method"] = pd.Categorical(rows["method"], categories=METHOD_ORDER, ordered=True)
    return rows.sort_values("method")


def select_walkthrough_sample(predictions_df):
    wide = prediction_wide(predictions_df)
    if "iou_baseline" in wide.columns and "iou_full" in wide.columns:
        wide["_delta"] = wide["iou_full"] - wide["iou_baseline"]
        candidates = wide[(wide["iou_full"] >= 0.5) & (wide["_delta"] > 0)]
        if len(candidates):
            return candidates.sort_values("_delta", ascending=False).iloc[0]
    iou_columns = [column for column in wide.columns if column.startswith("iou_")]
    wide["_best_iou"] = wide[iou_columns].max(axis=1)
    return wide.sort_values("_best_iou", ascending=False).iloc[0]


def draw_timeline(ax, rows, sample):
    apply_axis_style(ax, grid_axis="x")
    ax.spines["left"].set_visible(False)
    labels = ["Ground Truth"] + [method_label(method) for method in rows["method"]]
    y_positions = np.arange(len(labels))[::-1]
    bar_height = 0.62
    gt_start = float(sample["gt_start"])
    gt_end = float(sample["gt_end"])
    max_time = gt_end
    ax.axvspan(gt_start, gt_end, color="#e45756", alpha=0.08)
    ax.broken_barh([(gt_start, gt_end - gt_start)], (y_positions[0] - bar_height / 2, bar_height), facecolors="#e45756", alpha=0.88)
    y_labels = ["Ground Truth"]
    for index, (_, row) in enumerate(rows.iterrows(), start=1):
        pred_start = float(row["pred_start"])
        pred_end = float(row["pred_end"])
        max_time = max(max_time, pred_end)
        method = row["method"]
        ax.broken_barh([(pred_start, max(0.0, pred_end - pred_start))], (y_positions[index] - bar_height / 2, bar_height), facecolors=METHOD_COLORS.get(method, "#999999"), alpha=0.80)
        y_labels.append(f"{method_label(method)}  IoU={row['iou']:.2f}")
    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels)
    ax.set_xlim(0, max_time * 1.08 + 0.5)
    ax.set_xlabel("Time (seconds)")


def plot_sample_walkthrough(predictions_df, sample=None):
    if predictions_df.empty:
        print("Main predictions are missing.")
        return None
    sample = select_walkthrough_sample(predictions_df) if sample is None else sample
    rows = sample_rows(predictions_df, sample)
    fig = plt.figure(figsize=(11.4, 6.6))
    grid = fig.add_gridspec(2, 2, height_ratios=[0.9, 2.2], width_ratios=[1.3, 1.0], hspace=0.45, wspace=0.28)
    ax_info = fig.add_subplot(grid[0, 0])
    ax_info.axis("off")
    ax_info.text(0, 0.88, "Input sample", fontsize=12, weight="bold")
    ax_info.text(0, 0.55, f"Video: {sample['video_id']}", fontsize=10)
    ax_info.text(0, 0.30, f"Query: {shorten(sample['query'], 74)}", fontsize=10)
    ax_info.text(0, 0.05, f"Ground truth: {sample['gt_start']:.2f}s - {sample['gt_end']:.2f}s", fontsize=10)

    ax_iou = fig.add_subplot(grid[0, 1])
    apply_axis_style(ax_iou, grid_axis="x")
    colors = [METHOD_COLORS.get(method, "#999999") for method in rows["method"]]
    ax_iou.barh([method_label(method) for method in rows["method"]], rows["iou"], color=colors, alpha=0.82)
    ax_iou.set_xlim(0, 1.0)
    ax_iou.set_xlabel("IoU")
    ax_iou.set_title("Method IoU", weight="bold")

    ax_timeline = fig.add_subplot(grid[1, :])
    draw_timeline(ax_timeline, rows, sample)
    ax_timeline.set_title("Predicted Temporal Segments", weight="bold", loc="left")
    fig.suptitle("Single-Sample Pipeline Walkthrough", fontsize=15, weight="bold", y=0.99)
    return fig

fig = plot_sample_walkthrough(main_predictions)
sample_walkthrough_path = save_figure(fig, "sample_walkthrough.png") if fig else None
if fig:
    display(fig)
    plt.close(fig)

## Figure 6: Video Frame Walkthrough

In [ ]:
def sample_times(start, end, count, duration):
    start = max(0.0, min(float(start), float(duration)))
    end = max(0.0, min(float(end), float(duration)))
    if end <= start:
        end = min(float(duration), start + 0.1)
    return np.linspace(start, end, count + 2)[1:-1]


def read_frames(video_id, spans, frames_per_span=4):
    video_path = VIDEO_DIR / f"{video_id}.mp4"
    if not video_path.exists():
        raise FileNotFoundError(video_path)
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_total / fps if fps > 0 else 0
    frame_rows = []
    for label, start, end, color in spans:
        row = []
        for time in sample_times(start, end, frames_per_span, duration):
            frame_index = min(max(int(round(time * fps)), 0), frame_total - 1)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
            ok, frame = cap.read()
            if ok:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            else:
                frame = np.zeros((224, 224, 3), dtype=np.uint8)
            row.append((float(time), frame))
        frame_rows.append((label, color, row))
    cap.release()
    return frame_rows


def frame_spans(predictions_df, sample):
    rows = sample_rows(predictions_df, sample)
    spans = [("Ground Truth", sample["gt_start"], sample["gt_end"], "#e45756")]
    rows = rows[rows["method"].isin(FRAME_METHODS)]
    for _, row in rows.iterrows():
        label = f"{method_label(row['method'])}\nIoU={row['iou']:.2f}"
        spans.append((label, row["pred_start"], row["pred_end"], METHOD_COLORS.get(row["method"], "#999999")))
    return spans


def plot_video_frame_walkthrough(predictions_df, sample=None, frames_per_span=4):
    if predictions_df.empty:
        print("Main predictions are missing.")
        return None
    sample = select_walkthrough_sample(predictions_df) if sample is None else sample
    frame_rows = read_frames(sample["video_id"], frame_spans(predictions_df, sample), frames_per_span=frames_per_span)
    fig, axes = plt.subplots(len(frame_rows), frames_per_span, figsize=(frames_per_span * 2.7, len(frame_rows) * 2.0))
    axes = np.asarray(axes).reshape(len(frame_rows), frames_per_span)
    for row_index, (label, color, frames) in enumerate(frame_rows):
        for col_index, (time, frame) in enumerate(frames):
            ax = axes[row_index, col_index]
            ax.imshow(frame)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f"{time:.1f}s", fontsize=10)
            for spine in ax.spines.values():
                spine.set_color(color)
                spine.set_linewidth(3)
            if col_index == 0:
                ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=10, labelpad=34)
    fig.suptitle(f"Frame Walkthrough: {sample['video_id']} - {shorten(sample['query'], 70)}", fontsize=14, weight="bold", y=0.995)
    fig.tight_layout(rect=[0.05, 0, 1, 0.96])
    return fig

fig = plot_video_frame_walkthrough(main_predictions)
video_frame_path = save_figure(fig, "video_frame_walkthrough.png") if fig else None
if fig:
    display(fig)
    plt.close(fig)

## Saved Figure Paths

In [ ]:
figure_paths = {
    "main_ablation_metrics": main_ablation_path,
    "backbone_replacement": backbone_path,
    "ood1_backbone_comparison": ood_path,
    "main_iou_delta_vs_baseline": iou_delta_path,
    "sample_walkthrough": sample_walkthrough_path,
    "video_frame_walkthrough": video_frame_path,
}
figure_paths